# 6. Consultas Finales

Propósito: Consultas de negocio sobre los datos gold.

In [1]:
# --- Bootstrap del entorno (Windows + VSCode) ---
# VSCode inyecta el .env del repo (con rutas Linux para JAVA_HOME/HADOOP_HOME)
# dentro del kernel; en Windows esas rutas no existen y Spark no arranca.
# Ademas los notebooks corren desde notebooks/, por lo que fijamos el cwd y el
# sys.path en la raiz del repo para que 'data/...' e 'import app' funcionen.
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if not os.path.isdir(os.environ.get("JAVA_HOME", "")):
    os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-21.0.11.10-hotspot"
_hadoop = os.environ.get("HADOOP_HOME", "")
if not (os.path.isabs(_hadoop) and os.path.isdir(_hadoop)):
    os.environ["HADOOP_HOME"] = str(ROOT / "lib" / "hadoop")

print("ROOT:", ROOT)
print("JAVA_HOME:", os.environ["JAVA_HOME"])

ROOT: d:\Universidad\CICLO_VII\BigData\Proyecto\EP-GDM-G6
JAVA_HOME: C:\Program Files\Java\jdk-21


In [2]:
from pyspark.sql import SparkSession, functions as F
from app.utils.spark import SparkClient

# Reutiliza la sesion activa del kernel; si no hay, crea una con la config
# Windows-correcta de SparkClient (rutas nativas Hadoop, memoria del driver).
spark = SparkSession.getActiveSession() or SparkClient().get_session()
gold_path = "data/gold"
silver_path = "data/silver"

In [3]:
# Recaudación por año
spark.sql(f"""
  SELECT dc.Anio, SUM(mi.MontoRecaudado) as TotalRecaudado
  FROM parquet.`{gold_path}/MART_INGRESOS_GEOGRAFICO.parquet` mi
  JOIN parquet.`{gold_path}/DIM_CALENDARIO.parquet` dc ON mi.AnioMes = dc.AnioMes
  GROUP BY dc.Anio ORDER BY dc.Anio
""").toPandas()

,Anio,TotalRecaudado
0,2021,38905182522.68
1,2022,43029388603.63
2,2023,39890404877.91
3,2024,42006267911.10


In [4]:
# Top 10 ejecutoras por recaudación
mart_ejec = spark.read.parquet(f"{gold_path}/MART_INGRESOS_EJECUTORA.parquet")
mart_ejec.groupBy("Ejecutora", "Departamento").agg(F.sum("MontoRecaudado").alias("Total")).orderBy(F.col("Total").desc()).show(10)

+--------------------+--------------------+--------------+
|           Ejecutora|        Departamento|         Total|
+--------------------+--------------------+--------------+
|MUNICIPALIDAD MET...|                LIMA|10663739861.73|
|MUNICIPALIDAD DIS...|              ANCASH| 4240983347.62|
|MUNICIPALIDAD DIS...|               CUSCO| 2216652500.87|
|MUNICIPALIDAD PRO...|              ANCASH| 1826922611.45|
|MUNICIPALIDAD PRO...|PROVINCIA CONSTIT...| 1752186749.63|
|MUNICIPALIDAD DIS...|                LIMA| 1552867517.84|
|MUNICIPALIDAD DIS...|              ANCASH| 1477164267.22|
|MUNICIPALIDAD DIS...|            AREQUIPA| 1452805772.52|
|MUNICIPALIDAD DIS...|            AREQUIPA| 1452655054.83|
|MUNICIPALIDAD PRO...|            AREQUIPA| 1241624794.89|
+--------------------+--------------------+--------------+
only showing top 10 rows


In [5]:
# Ejecución por departamento
mart_geo = spark.read.parquet(f"{gold_path}/MART_INGRESOS_GEOGRAFICO.parquet")
mart_geo.groupBy("Departamento").agg(
    F.sum("MontoPIA").alias("PIA"),
    F.sum("MontoPIM").alias("PIM"),
    F.sum("MontoRecaudado").alias("Recaudado"),
    F.avg("PctEjecucion").alias("PctPromedio")
).orderBy(F.col("Recaudado").desc()).toPandas()

,Departamento,PIA,PIM,Recaudado,PctPromedio
0,LIMA,19156930710,27683900364,30151868814.98,370.297401
1,CUSCO,10277182010,17681013029,18689458518.39,19.675370
2,ANCASH,6786597809,16463157341,17219813826.65,-6.802901
3,AREQUIPA,5332843430,12289718254,14301276500.09,7.308497
4,PIURA,5035997166,10614084003,9757479514.66,1489.684493
5,LA LIBERTAD,4042577950,8536078784,8390106637.59,22.098881
6,CAJAMARCA,3835438061,7272895119,6936795689.07,525.819176
7,PUNO,3168513552,5438776333,5488495136.61,11.716161
8,ICA,2734688980,5666529688,5485910920.72,13.087561
9,JUNIN,2691501267,5068539192,5323239949.38,11.758831


In [6]:
# Deuda predial por municipio
mart_predial = spark.read.parquet(f"{gold_path}/MART_PREDIAL.parquet")
mart_predial.filter(F.col("PreguntaDescripcion").contains("deuda")).groupBy("Municipalidad").agg(
    F.avg("ValorNumerico").alias("DeudaPromedio")
).orderBy(F.col("DeudaPromedio").desc()).show(10)

+--------------------+---------------+
|       Municipalidad|  DeudaPromedio|
+--------------------+---------------+
|MUNICIPALIDAD DIS...|30707282.878261|
|MUNICIPALIDAD DIS...|12130020.227000|
|MUNICIPALIDAD DIS...| 6833249.703250|
|MUNICIPALIDAD DIS...| 5822655.744000|
|MUNICIPALIDAD DIS...| 3896098.131143|
|MUNICIPALIDAD DIS...| 2656811.150250|
|MUNICIPALIDAD DIS...| 2123712.349750|
|MUNICIPALIDAD DIS...| 1912012.659250|
|MUNICIPALIDAD PRO...| 1496868.289750|
|MUNICIPALIDAD PRO...| 1376381.239750|
+--------------------+---------------+
only showing top 10 rows


In [7]:
# Cobertura de servicios RENAMU
mart_renamu = spark.read.parquet(f"{gold_path}/MART_RENAMU.parquet")
mart_renamu.filter(F.col("EsAfirmativo") == True).groupBy("Departamento", "Descripcion").agg(
    F.count("*").alias("MunicipiosConServicio")
).orderBy("Departamento", F.col("MunicipiosConServicio").desc()).show(20)

+------------+-----------+---------------------+
|Departamento|Descripcion|MunicipiosConServicio|
+------------+-----------+---------------------+
|    AMAZONAS|    VFI_P13|                  420|
|    AMAZONAS|    VFI_P85|                  420|
|    AMAZONAS|    VFI_P19|                  420|
|    AMAZONAS|    VFI_P80|                  420|
|    AMAZONAS|    VFI_P79|                  420|
|    AMAZONAS|    VFI_P23|                  420|
|    AMAZONAS|    VFI_P30|                  420|
|    AMAZONAS|    VFI_P69|                  420|
|    AMAZONAS|    VFI_P24|                  420|
|    AMAZONAS|    VFI_P21|                  420|
|    AMAZONAS|    VFI_P12|                  420|
|    AMAZONAS|    VFI_P62|                  420|
|    AMAZONAS|     P60A_1|                  420|
|    AMAZONAS|    VFI_P14|                  420|
|    AMAZONAS|    VFI_P67|                  420|
|    AMAZONAS|    VFI_P58|                  420|
|    AMAZONAS|    VFI_P46|                  420|
|    AMAZONAS|    VF